<a href="https://colab.research.google.com/github/Buddhiimz/DeepLearning_Project/blob/feature%2Fbuddhima/Proj_InceptionV3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# 0.1 Check TF and GPU
import tensorflow as tf
print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

TF: 2.19.0
GPUs: []


In [13]:
# 1. Imports and global config
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input

In [14]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
# 1. Global settings
TRAIN_DIR = "/content/drive/MyDrive/DL Project/Flowers train"
TEST_DIR  = "/content/drive/MyDrive/DL Project/Flowers test"

# InceptionV3 prefers 299x299 input
IMG_SIZE = (299, 299)   # InceptionV3 default input size
BATCH_SIZE = 16         # lower if OOM
SEED = 42

In [18]:
# # Human-readable label mapping
# 2. Map folder codes to full species names (from your list)
orchid_label_map = {
    "n0000": "Anoectochilus burmanicus rolfe",
    "n0001": "Bulbophyllum auricomum lindl",
    "n0002": "Bulbophyllum dayanum rchb",
    "n0003": "Bulbophyllum lasiochilum par. & rchb",
    "n0004": "Bulbophyllum limbatum",
    "n0005": "Bulbophyllum longissimum (ridl.) ridl",
    "n0006": "Bulbophyllum medusae (lindl.) rchb",
    "n0007": "Bulbophyllum patens king ex hk.f.",
    "n0008": "Bulbophyllum rufuslabram",
    "n0009": "Bulbophyllum siamensis rchb",
    "n0010": "Calenthe rubens",
    "n0011": "Chiloschista parishii seidenf.",
    "n0012": "Chiloschista viridiflora seidenf.",
    "n0013": "Cymbidium aloifolium (l.) sw.",
    "n0014": "Dendrobium chrysotoxum lindl",
    "n0015": "Dendrobium farmeri paxt.",
    "n0016": "Dendrobium fimbriatum hook",
    "n0017": "Dendrobium lindleyi steud",
    "n0018": "Dendrobium pulchellum roxb",
    "n0019": "Dendrobium pulchellum",
    "n0020": "Dendrobium secundum bl-lindl",
    "n0021": "Dendrobium senile par. & rchb.f.",
    "n0022": "Dendrobium signatum rchb. f",
    "n0023": "Dendrobium thyrsiflorum rchb. f.",
    "n0024": "Dendrobium tortile lindl",
    "n0025": "Dendrobium tortile",
    "n0026": "Hygrochillus parishii var. marrioftiana (rchb.f.)",
    "n0027": "Paphiopedilum bellatulum",
    "n0028": "Paphiopedilum callosum",
    "n0029": "Paphiopedilum charlesworthii",
    "n0030": "Paphiopedilum concolor",
    "n0031": "Paphiopedilum exul",
    "n0032": "Paphiopedilum godefroyae",
    "n0033": "Paphiopedilum gratrixianum",
    "n0034": "Paphiopedilum henryanum",
    "n0035": "Paphiopedilum intanon-villosum",
    "n0036": "Paphiopedilum niveum (rchb.f.) stein",
    "n0037": "Paphiopedilum parishii",
    "n0038": "Paphiopedilum spicerianum",
    "n0039": "Paphiopedilum sukhakulii",
    "n0040": "Pelatantheria bicuspidata (rolfe ex downie) tang & wang",
    "n0041": "Pelatantheria insectiflora (rchb.f.) ridl.",
    "n0042": "Phaius tankervilleae (banks ex i' heritier) blume",
    "n0043": "Phalaenopsis cornucervi (breda) bl. & rchb.f.",
    "n0044": "Rhynchostylis gigantea (lindl.) ridl.",
    "n0045": "Trichoglottis orchideae (koern) garay.",
    "n0046": "Bulbophyllum auratum Lindl.",
    "n0047": "Bulbophyllum morphologorum F.Kranzl.",
    "n0048": "Dendrobium cumulatum Lindl.",
    "n0049": "Maxiralia tenui folia",
    "n0050": "Paphiopedilum vejvarutianum O. Gruss & Roellke",
    "n0051": "Oncidium goldiana"
}


In [19]:
# Data Generators
# 3. Create ImageDataGenerators using Inception preprocess_input
train_aug = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # scale to Inception's [-1,1]
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_aug = ImageDataGenerator(preprocessing_function=preprocess_input)

# 3.1 Flow from directory (train & test)
train_gen = train_aug.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

test_gen = test_aug.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# 3.2 Build readable class name list (index->human name)
index_to_folder = {v: k for k, v in train_gen.class_indices.items()}
class_names = [orchid_label_map.get(index_to_folder[i], index_to_folder[i])
               for i in range(len(index_to_folder))]

print("Detected class folders (codes):", list(train_gen.class_indices.keys())[:6], "...")
print("Readable class names (first 6):", class_names[:6], "...")
print("Train samples:", train_gen.samples, " | Test samples:", test_gen.samples)


Found 2834 images belonging to 52 classes.
Found 745 images belonging to 52 classes.
Detected class folders (codes): ['n0000', 'n0001', 'n0002', 'n0003', 'n0004', 'n0005'] ...
Readable class names (first 6): ['Anoectochilus burmanicus rolfe', 'Bulbophyllum auricomum lindl', 'Bulbophyllum dayanum rchb', 'Bulbophyllum lasiochilum par. & rchb', 'Bulbophyllum limbatum', 'Bulbophyllum longissimum (ridl.) ridl'] ...
Train samples: 2834  | Test samples: 745


In [20]:
# 4. Compute class weights to help with imbalance (recommended)
y_train = train_gen.classes
unique = np.unique(y_train)
class_weights_values = compute_class_weight(class_weight='balanced', classes=unique, y=y_train)
class_weight = {i: w for i, w in enumerate(class_weights_values)}
print("Class weights:", class_weight)

Class weights: {0: np.float64(1.1354166666666667), 1: np.float64(1.1354166666666667), 2: np.float64(2.477272727272727), 3: np.float64(1.1122448979591837), 4: np.float64(3.0277777777777777), 5: np.float64(1.8793103448275863), 6: np.float64(2.369565217391304), 7: np.float64(3.40625), 8: np.float64(2.18), 9: np.float64(1.2674418604651163), 10: np.float64(2.8684210526315788), 11: np.float64(0.5860215053763441), 12: np.float64(1.0092592592592593), 13: np.float64(0.7785714285714286), 14: np.float64(1.09), 15: np.float64(1.2674418604651163), 16: np.float64(0.8257575757575758), 17: np.float64(0.3784722222222222), 18: np.float64(0.5240384615384616), 19: np.float64(1.0686274509803921), 20: np.float64(0.592391304347826), 21: np.float64(1.9464285714285714), 22: np.float64(2.0961538461538463), 23: np.float64(2.477272727272727), 24: np.float64(0.3303030303030303), 25: np.float64(2.8684210526315788), 26: np.float64(1.7580645161290323), 27: np.float64(0.6055555555555555), 28: np.float64(0.939655172413

In [21]:
# 5. Build model function that returns both base_model and full_model
def build_inception_transfer(num_classes, input_shape=(299,299,3), dropout_rate=0.5):
    # Load pretrained InceptionV3 (ImageNet), exclude top classifier
    base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False  # freeze base initially

    # Build custom head on top
    x = base_model.output
    x = layers.GlobalAveragePooling2D(name="avg_pool")(x)
    x = layers.Dropout(dropout_rate, name="top_dropout")(x)
    x = layers.Dense(512, activation='relu', name="fc1")(x)
    x = layers.BatchNormalization(name="bn_fc1")(x)
    x = layers.Dropout(0.3, name="drop_fc1")(x)
    outputs = layers.Dense(num_classes, activation='softmax', name="predictions")(x)

    model = models.Model(inputs=base_model.input, outputs=outputs, name="InceptionV3_transfer")
    return base_model, model

# Construct model
base_model, model = build_inception_transfer(num_classes=len(class_names), input_shape=(IMG_SIZE[0],IMG_SIZE[1],3))
model.summary()


87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "InceptionV3_transfer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 299, 299,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 149, 149,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 149, 149,  │         96 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 149, 149,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 147, 147,  │      9,216 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 147, 147,  │         96 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 147, 147,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 147, 147,  │     18,432 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 147, 147,  │        192 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 147, 147,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 73, 73,    │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 73, 73,    │      5,120 │ max_pooling2d[0]… │
│                     │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 73, 73,    │        240 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 73, 73,    │          0 │ batch_normalizat… │
│ (Activation)        │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 71, 71,    │    138,240 │ activation_3[0][… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 71, 71,    │        576 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 71, 71,    │          0 │ batch_normalizat

 Total params: 22,880,596 (87.28 MB)

 Trainable params: 1,076,788 (4.11 MB)

 Non-trainable params: 21,803,808 (83.17 MB)

In [22]:
# 6. Compile for head training
initial_lr = 1e-3
model.compile(
    optimizer=optimizers.Adam(learning_rate=initial_lr),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks: save best, reduce LR, early stop
ckpt_head = callbacks.ModelCheckpoint("inception_head_best.h5", monitor='val_loss', save_best_only=True, verbose=1)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1, min_lr=1e-7)
earlystop = callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)


In [ ]:
# 7. Train only the new head (base_model frozen)
EPOCHS_HEAD = 8  # a reasonable quick first stage; increase if desired

history_head = model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=EPOCHS_HEAD,
    callbacks=[ckpt_head, reduce_lr, earlystop],
    class_weight=class_weight  # optional but recommended
)


In [ ]:
# 8. Fine-tuning: unfreeze top layers of the base model
# Strategy: unfreeze the last N layers of base_model for fine-tuning
N = 50  # start with 50; increase/decrease depending on dataset & GPU
print("Total base layers:", len(base_model.layers))
for layer in base_model.layers[:-N]:
    layer.trainable = False
for layer in base_model.layers[-N:]:
    layer.trainable = True

# Recompile with a much lower learning rate for fine-tuning
fine_lr = 1e-5
model.compile(
    optimizer=optimizers.Adam(learning_rate=fine_lr),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks & checkpoint for fine-tuning
ckpt_ft = callbacks.ModelCheckpoint("inception_finetune_best.h5", monitor='val_loss', save_best_only=True, verbose=1)

# 8.1 Train (fine-tune)
EPOCHS_FINETUNE = 12  # tune as needed
history_fine = model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=EPOCHS_FINETUNE,
    callbacks=[ckpt_ft, reduce_lr, earlystop],
    class_weight=class_weight
)


In [ ]:
# 9. Load best fine-tuned weights (if saved)
if os.path.exists("inception_finetune_best.h5"):
    model.load_weights("inception_finetune_best.h5")
    print("Loaded best fine-tuned weights.")

# 9.1 Evaluate on test set
test_loss, test_acc = model.evaluate(test_gen, verbose=1)
print(f"Final Test Accuracy: {test_acc*100:.2f}%, Test Loss: {test_loss:.4f}")

# 9.2 Save a combined history for plotting (concatenate histories)
# (optional) combine history dictionaries for plotting
def combine_histories(h1, h2):
    combined = {}
    for k in h1.history:
        combined[k] = h1.history[k] + h2.history.get(k, [])
    # include keys present only in h2
    for k in h2.history:
        if k not in combined:
            combined[k] = h2.history[k]
    return combined

combined_history = combine_histories(history_head, history_fine) if ('history_head' in globals() and 'history_fine' in globals()) else (history_head.history if 'history_head' in globals() else history_fine.history)


In [ ]:
# 10. Plot training/validation accuracy & loss (uses combined_history)
hist = combined_history if isinstance(combined_history, dict) else combined_history  # ensure dict-like

# extract arrays for plotting
def ensure_list(x):
    return x if isinstance(x, list) else list(x)

acc = hist['accuracy']
val_acc = hist['val_accuracy']
loss = hist['loss']
val_loss = hist['val_loss']

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(acc, label='train_acc')
plt.plot(val_acc, label='val_acc')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(loss, label='train_loss')
plt.plot(val_loss, label='val_loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# 11. Predictions for confusion matrix and classification report
y_probs = model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_probs, axis=1)
y_true = test_gen.classes

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14,12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix (InceptionV3)')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.show()

# Classification report with readable names
print("Classification report:\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))


In [ ]:
# 12. Save model and metrics
model.save("inception_orchid_final.h5")
print("Saved model to inception_orchid_final.h5")

# Save metrics (test acc/loss + last epoch train/val)
import json
metrics = {
    "test_accuracy": float(test_acc),
    "test_loss": float(test_loss),
    "num_classes": len(class_names),
    "train_samples": int(train_gen.samples),
    "test_samples": int(test_gen.samples)
}
with open("inception_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved metrics to inception_metrics.json")


In [ ]:
# 13. Predict a single image (top-k), display image + readable outputs
from tensorflow.keras.preprocessing import image

def predict_single_inception(img_path, model, IMG_SIZE, class_names, top_k=3):
    # Load image and resize to IMG_SIZE
    img = image.load_img(img_path, target_size=IMG_SIZE)
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)  # must use same preprocessing as training

    # Predict
    probs = model.predict(x)[0]  # probabilities
    top_idx = probs.argsort()[-top_k:][::-1]

    print("Top predictions:")
    for i in top_idx:
        print(f"  {class_names[i]} : {probs[i]*100:.2f}%")

    # Show original (non-preprocessed) image for clarity
    plt.imshow(image.array_to_img(image.img_to_array(img)/255.0))
    plt.axis('off')
    plt.title(f"Top: {class_names[top_idx[0]]} ({probs[top_idx[0]]*100:.1f}%)")
    plt.show()

# Example usage:
# predict_single_inception("/content/my_orchid.jpg", model, IMG_SIZE, class_names, top_k=3)
